# Experiment 1 — Authority direction (§4)

Parametrized notebook: set MODEL_KEY in the config cell, then Run All on an A100.

For Exp 2-4, make sure Experiment 1 has already been run for the chosen model (the result bundle is read from Drive).


In [ ]:
# Clone repo (for data/ + tests/) and install it in editable mode
import os, sys, importlib, site
REPO_DIR = "/content/Mech_spoof"

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ChuloIva/Mech_spoof.git {REPO_DIR}

!pip install -q -e {REPO_DIR}

# Make sure mech_spoof resolves data/ to the cloned repo
os.environ["MECH_SPOOF_ROOT"] = REPO_DIR

# Ensure the current kernel picks up the freshly installed package
# (Colab caches sys.path before pip runs; editable installs need a nudge)
src_path = f"{REPO_DIR}/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
site.main()

import mech_spoof
print("mech_spoof loaded from:", mech_spoof.__file__)

In [ ]:
# Auth + Drive
# Paste your HF token here (leave as "" to skip; needed for gated models like Llama-3, Gemma, Mistral)
HF_TOKEN = ""
ANTHROPIC_API_KEY = ""  # optional, only used by some eval/dataset-gen paths

import os
from google.colab import drive
from huggingface_hub import login as hf_login

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    hf_login(token=HF_TOKEN, add_to_git_credential=False)

if ANTHROPIC_API_KEY:
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY

drive.mount('/content/drive')

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = 'qwen'            # qwen | llama3 | mistral | gemma | phi3
EXPERIMENT = 1
DRIVE_ROOT = Path('/content/drive/MyDrive/mech_spoof_results')

slug = MODEL_CONFIGS[MODEL_KEY].slug
exp_dirname = {1: 'exp1_authority', 2: 'exp2_conflict', 3: 'exp3_refusal', 4: 'exp4_attacks'}[EXPERIMENT]
OUT_DIR = DRIVE_ROOT / slug / exp_dirname
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

In [ ]:
# Run the experiment
if EXPERIMENT == 1:
    from mech_spoof.experiments.exp1_authority import run_experiment_1
    result = run_experiment_1(MODEL_KEY, OUT_DIR)
elif EXPERIMENT == 2:
    from mech_spoof.experiments.exp2_conflict import run_experiment_2
    result = run_experiment_2(MODEL_KEY, OUT_DIR, exp1_dir=DRIVE_ROOT / slug / 'exp1_authority')
elif EXPERIMENT == 3:
    from mech_spoof.experiments.exp3_refusal import run_experiment_3
    result = run_experiment_3(MODEL_KEY, OUT_DIR, exp1_dir=DRIVE_ROOT / slug / 'exp1_authority')
elif EXPERIMENT == 4:
    from mech_spoof.experiments.exp4_attacks import run_experiment_4
    result = run_experiment_4(
        MODEL_KEY, OUT_DIR,
        exp1_dir=DRIVE_ROOT / slug / 'exp1_authority',
        exp3_dir=DRIVE_ROOT / slug / 'exp3_refusal',
    )
print('done:', OUT_DIR)

In [ ]:
# Inspect + plot (Exp 1 only — per-model plot)
if EXPERIMENT == 1:
    from mech_spoof.viz import plot_layer_accuracy
    plot_layer_accuracy(result.accuracies, MODEL_KEY, OUT_DIR / 'layer_accuracy.png')
    print({l: round(a, 3) for l, a in result.accuracies.items()})